# LSTM

In [ ]:
# LSTM import
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM

from tensorflow.keras.layers import Input, Dense, LSTM, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# Label encoding
from sklearn.preprocessing import LabelEncoder

In [86]:
filepath_train = 'data/ais_train.csv'
filepath_test = 'data/ais_test.csv'

# Load AIS historical data
train = pd.read_csv(filepath_train, sep ='|')  # Replace with your dataset
test = pd.read_csv(filepath_test, sep = ',')

In [87]:
# Replace special values with NaN
train.replace({'rot': {127: np.nan, -127: np.nan, -128: np.nan},
               'sog': {102.3: np.nan},
               'cog': {360: np.nan},
               'heading': {511: np.nan}}, inplace=True)

# drop columns except vesselId, sog, cog, heading, rot, time, latitude, longitude
train = train[['time','vesselId', 'sog', 'cog', 'heading', 'rot', 'latitude', 'longitude']]

# Convert time to datetime
train['time'] = pd.to_datetime(train['time'])

# Time difference between two consecutive rows in seconds for each vessel
train['time_diff'] = train.groupby('vesselId')['time'].diff().dt.total_seconds()

# Label encoding for vesselId
LabelEncoder_vesselId = LabelEncoder()
train['vesselId'] = LabelEncoder_vesselId.fit_transform(train['vesselId'])

# drop rows with NaN values
train = train.dropna()

# Scale features with min-max scaler individually

scaler_sog = MinMaxScaler(feature_range=(0, 1))
train['sog'] = scaler_sog.fit_transform(train[['sog']])

scaler_cog = MinMaxScaler(feature_range=(0, 1))
train['cog'] = scaler_cog.fit_transform(train[['cog']])

scaler_heading = MinMaxScaler(feature_range=(0, 1))
train['heading'] = scaler_heading.fit_transform(train[['heading']])

scaler_rot = MinMaxScaler(feature_range=(0, 1))
train['rot'] = scaler_rot.fit_transform(train[['rot']])

scaler_latitude = MinMaxScaler(feature_range=(0, 1))
train['latitude'] = scaler_latitude.fit_transform(train[['latitude']])

scaler_longitude = MinMaxScaler(feature_range=(0, 1))
train['longitude'] = scaler_longitude.fit_transform(train[['longitude']])

scaler_time_diff = MinMaxScaler(feature_range=(0, 1))
train['time_diff'] = scaler_time_diff.fit_transform(train[['time_diff']])

display(train)

,time,vesselId,sog,cog,heading,rot,latitude,longitude,time_diff
143,2024-01-01 00:17:17,71,0.000978,0.205335,0.248447,0.543307,0.175773,0.926170,0.000061
145,2024-01-01 00:18:57,175,0.000978,0.418727,0.519669,0.496063,0.728315,0.529692,0.000027
146,2024-01-01 00:19:39,211,0.126223,0.156710,0.115942,0.433071,0.689652,0.861753,0.000056
148,2024-01-01 00:20:35,682,0.000000,0.318144,0.569358,0.496063,0.770389,0.472700,0.000121
149,2024-01-01 00:21:37,161,0.000000,0.584607,0.252588,0.476378,0.747052,0.568382,0.000060
...,...,...,...,...,...,...,...,...,...
1522060,2024-05-07 23:59:07,682,0.131115,0.997777,0.002070,0.496063,0.844476,0.466928,0.000214
1522061,2024-05-07 23:59:08,85,0.167319,0.034176,0.026915,0.496063,0.732443,0.449076,0.000210
1522062,2024-05-07 23:59:08,459,0.145793,0.749653,0.559006,0.492126,0.823495,0.468665,0.000210
1522063,2024-05-07 23:59:08,596,0.182975,0.022228,0.012422,0.496063,0.726664,0.514871,0.000254


In [88]:
# Example structure
# Multi-index with vessel_id and datetime
# vessel_id | datetime           | feature_1 | feature_2 | longitude | latitude
# ---------------------------------------------------------------------------
# 001       | 2023-01-01 00:00:00 | ...       | ...       | ...       | ...
# 001       | 2023-01-01 01:00:00 | ...       | ...       | ...       | ...
# 002       | 2023-01-01 00:00:00 | ...       | ...       | ...       | ...

# Separate DataFrames for each vessel
vessel_data = {
    vessel_id: data
    for vessel_id, data in train.groupby('vesselId')
}



In [89]:
from tqdm import tqdm


# Parameters
sequence_length = 168  # Length of history for each sequence (adjust as needed)

# Lists to collect all sequences and targets across vessels
all_X = []
all_y = []
# Process each vessel's DataFrame with a progress bar
for vessel_id, df_vessel in tqdm(vessel_data.items(), desc="Processing vessels"):
    input_sequences = []
    target_coordinates = []
    
    # Generate sequences for the current vessel
    for i in range(len(df_vessel) - sequence_length):
        # Extract the input sequence (e.g., features)
        input_seq = df_vessel.iloc[i:i+sequence_length][['vesselId', 'time_diff', 'sog', 'cog', 'heading', 'rot', 'longitude', 'latitude']].values
        input_sequences.append(input_seq)
        
        # Extract the target (longitude, latitude at the next timestep)
        target_coord = df_vessel.iloc[i + sequence_length][['sog', 'cog', 'heading', 'rot', 'longitude', 'latitude']].values
        target_coordinates.append(target_coord)
    
    # Extend to the global list of sequences and targets
    all_X.extend(input_sequences)
    all_y.extend(target_coordinates)

# Convert lists to arrays for model input
X_train = np.array(all_X)  # Shape: (total_sequences, sequence_length, num_features)
y_train = np.array(all_y)  # Shape: (total_sequences, num_targets)

# Now X_train and y_train are ready to be used for training the deep learning model.


Processing vessels: 100%|██████████| 686/686 [31:31<00:00,  2.76s/it]  


In [90]:
display(X_train.shape)
display(y_train.shape)

(1374232, 168, 8)

(1374232, 6)

In [ ]:
# Ensure the numpy arrays are of type float32
X_train = X_train.astype(np.float32)
y_train = y_train.astype(np.float32)

# Split y_train into separate targets for each head
y_train_lat_lon = y_train[:, -2:]  # Last 2 columns for latitude and longitude
y_train_other_features = y_train[:, :-2]  # All columns except the last 2 for SOG, COG, heading, and ROT


# Define the input layer
input_layer = Input(shape=(sequence_length, X_train.shape[2]))

# Shared LSTM layers
x = LSTM(64, return_sequences=True)(input_layer)
x = Dropout(0.2)(x)
x = LSTM(32, return_sequences=False)(x)
x = Dropout(0.2)(x)

# Separate heads for different targets
# Head for longitude and latitude
lat_lon_output = Dense(2, name='lat_lon')(x)

# Head for SOG, COG, heading, and ROT
other_features_output = Dense(4, name='other_features')(x)

# Define the model with multiple outputs
model = Model(inputs=input_layer, outputs=[lat_lon_output, other_features_output])

# Compile with separate losses and optional weighting
model.compile(
    optimizer='sgd',
    loss={'lat_lon': 'mse', 'other_features': 'mse'},
    loss_weights={'lat_lon': 1.0, 'other_features': 0.5}  # Optional weighting
)

model.summary()

# Set up early stopping to monitor the validation loss
early_stopping = EarlyStopping(
    monitor='val_loss',       # Monitor the overall validation loss
    patience=2,               # Stop training after 5 epochs with no improvement
    restore_best_weights=True  # Restore the best weights at the end
)

# Train the model with early stopping
history = model.fit(
    X_train, 
    {'lat_lon': y_train_lat_lon, 'other_features': y_train_other_features},  # Separate targets for each head
    epochs=5,
    batch_size=62,
    validation_split=0.2,
    callbacks=[early_stopping]
)

# Plot the training history
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.show()


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 168, 8)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_8 (LSTM)       │ (None, 168, 64)   │     18,688 │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 168, 64)   │          0 │ lstm_8[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_9 (LSTM)       │ (None, 32)        │     12,416 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 32)        │          0 │ lstm_9[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lat_lon (Dense)     │ (None, 2)         │         66 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ other_features      │ (None, 4)         │        132 │ dropout_5[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 31,302 (122.27 KB)

 Trainable params: 31,302 (122.27 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
 6031/17733 ━━━━━━━━━━━━━━━━━━━━ 4:22:53 1s/step - lat_lon_loss: 0.0206 - loss: 0.0365 - other_features_loss: 0.0159

KeyboardInterrupt: 

In [ ]:
# Save the model
model.save('my_model.keras')


In [ ]:
# Load the model
model = tf.keras.models.load_model('my_model.keras')

c:\Users\avira\OneDrive - NTNU\Master EMIL\1. semester\ML\TDT4173-Gruppe-8\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
# Assume the training data is stored as a dictionary with vessel IDs as keys
# Each entry (vessel ID) contains a DataFrame with past data
# vessel_data = {'001': df_vessel_001, '002': df_vessel_002, ...}

# Dictionary to store each vessel's last sequence of data from training
last_known_sequences = {}

for vessel_id, df_vessel in vessel_data.items():
    # Get the last `sequence_length` rows for this vessel
    last_sequence = df_vessel.iloc[-sequence_length:][['sog', 'cog', 'heading', 'rot']].values
    last_known_sequences[vessel_id] = last_sequence

# Now `last_known_sequences` contains the last known sequence for each vessel in the training data

#display(last_known_sequences)

In [ ]:
evaluation_set = pd.read_csv(filepath_test, sep = ',')  # Replace with your dataset

# Convert time to datetime
evaluation_set['time'] = pd.to_datetime(evaluation_set['time'])

# Label encoding for vesselId
evaluation_set['vesselId'] = LabelEncoder_vesselId.transform(evaluation_set['vesselId'])

display(evaluation_set)

,ID,vesselId,time,scaling_factor
0,0,84,2024-05-08 00:03:16,0.3
1,1,623,2024-05-08 00:06:17,0.3
2,2,596,2024-05-08 00:10:02,0.3
3,3,542,2024-05-08 00:10:34,0.3
4,4,1,2024-05-08 00:12:27,0.3
...,...,...,...,...
51734,51734,48,2024-05-12 23:59:58,0.1
51735,51735,110,2024-05-12 23:59:58,0.1
51736,51736,610,2024-05-12 23:59:58,0.1
51737,51737,574,2024-05-12 23:59:58,0.1


In [ ]:
from tqdm import tqdm

# Dictionary to store predictions for each vessel in the evaluation set
predictions = []

# Iterate over each vessel in the evaluation set with a progress bar
for vessel_id in tqdm(evaluation_set['vesselId'].unique(), desc="Predicting for vessels"):
    if vessel_id in vessel_data:
        # Get the historical data for the vessel
        df_vessel = vessel_data[vessel_id]
        
        # Scale the features of the vessel
        scaled_features = df_vessel[['sog', 'cog', 'heading', 'rot']]
        scaled_targets = df_vessel[['longitude', 'latitude']]
        
        # Prepare the DataFrame with scaled features and targets
        df_vessel_scaled = pd.DataFrame(scaled_features, columns=['sog', 'cog', 'heading', 'rot'], index=df_vessel.index)
        df_vessel_scaled['longitude'] = scaled_targets['longitude']
        df_vessel_scaled['latitude'] = scaled_targets['latitude']
        
        # Extract the most recent `sequence_length` data points for input
        latest_sequence = df_vessel_scaled.iloc[-sequence_length:][['sog', 'cog', 'heading', 'rot']].values

        # Ensure the sequence is the right shape for input to the model
        current_sequence = latest_sequence.reshape(1, sequence_length, -1)

        # Get all future timestamps for the vessel in the evaluation set
        vessel_timestamps = evaluation_set[evaluation_set['vesselId'] == vessel_id]['time']

        # Predict sequentially for each future timestamp
        for timestamp in vessel_timestamps:
            # Predict the next longitude and latitude
            predicted_scaled = model.predict(current_sequence, verbose=0)
            
            # Inverse transform to get original scale
            temp_array = np.zeros((1, 6))  # Create a temporary array with the same number of features
            temp_array[0, :2] = predicted_scaled  # Place the predicted coordinates in the correct position
            predicted_coordinates = scaler.inverse_transform(temp_array)
            predictions.append({
                'vessel_id': vessel_id,
                'timestamp': timestamp,
                'predicted_longitude': predicted_coordinates[0][0],
                'predicted_latitude': predicted_coordinates[0][1]
            })
            
            # Update the sequence for the next prediction step
            # Create a new input sequence by appending the prediction and removing the oldest step
            new_features = current_sequence[0, 1:, :]  # Remove the oldest step
            new_step = np.zeros((1, 4))  # Create a new step with the same number of features
            new_step[0, :2] = predicted_scaled  # Place the predicted coordinates in the correct position
            new_features = np.vstack((new_features, new_step))  # Append the new prediction
            current_sequence = new_features.reshape(1, sequence_length, -1)

# Convert predictions to a DataFrame for easy interpretation
predictions_df = pd.DataFrame(predictions)

# Display the prediction results
predictions_df


Predicting for vessels: 100%|██████████| 215/215 [1:07:44<00:00, 18.90s/it]


,vessel_id,timestamp,predicted_longitude,predicted_latitude
0,84,2024-05-08 00:03:16,51.088003,296.480117
1,84,2024-05-08 00:36:14,57.701802,297.485301
2,84,2024-05-08 00:51:12,56.356230,311.839392
3,84,2024-05-08 01:12:14,56.592925,323.865222
4,84,2024-05-08 01:30:14,51.760816,305.285422
...,...,...,...,...
51734,539,2024-05-12 22:25:40,50.142694,289.239091
51735,539,2024-05-12 22:55:40,50.139694,289.233578
51736,539,2024-05-12 23:13:40,50.126786,289.172291
51737,539,2024-05-12 23:34:40,50.109939,289.077582


In [ ]:
# sort after the evaluation set by merging vesselId and time
# sort index

display(predictions_df)

,vessel_id,timestamp,predicted_longitude,predicted_latitude
0,84,2024-05-08 00:03:16,51.088003,296.480117
233,623,2024-05-08 00:06:17,55.545195,263.889775
410,596,2024-05-08 00:10:02,53.899362,291.723757
547,542,2024-05-08 00:10:34,47.293317,235.181100
697,1,2024-05-08 00:12:27,50.122165,271.294520
...,...,...,...,...
33502,332,2024-05-12 23:59:58,50.167249,289.252842
50253,48,2024-05-12 23:59:58,50.169606,289.256553
9865,82,2024-05-12 23:59:58,50.167188,289.252692
41980,574,2024-05-12 23:59:58,50.166171,289.251834


In [ ]:
# Make copy of predictions_df
predictions_df_copy = predictions_df.copy()

# Rename Vessel_id to vesselId
predictions_df_copy.rename(columns={'vessel_id': 'vesselId'}, inplace=True)
predictions_df_copy.rename(columns={'timestamp': 'time'}, inplace=True)
predictions_df_copy.rename(columns={'predicted_longitude': 'longitude_predicted'}, inplace=True)
predictions_df_copy.rename(columns={'predicted_latitude': 'latitude_predicted'}, inplace=True)

# Merge evaluation set with predictions_df
predictions_df_copy = evaluation_set.merge(predictions_df_copy, on=['vesselId', 'time'], how='left')

display(predictions_df_copy)



,ID,vesselId,time,scaling_factor,longitude_predicted,latitude_predicted
0,0,84,2024-05-08 00:03:16,0.3,51.088003,296.480117
1,1,623,2024-05-08 00:06:17,0.3,55.545195,263.889775
2,2,596,2024-05-08 00:10:02,0.3,53.899362,291.723757
3,3,542,2024-05-08 00:10:34,0.3,47.293317,235.181100
4,4,1,2024-05-08 00:12:27,0.3,50.122165,271.294520
...,...,...,...,...,...,...
51734,51734,48,2024-05-12 23:59:58,0.1,50.169606,289.256553
51735,51735,110,2024-05-12 23:59:58,0.1,50.168595,289.257068
51736,51736,610,2024-05-12 23:59:58,0.1,50.166183,289.245184
51737,51737,574,2024-05-12 23:59:58,0.1,50.166171,289.251834


In [ ]:
# To csv file in format ID,longitude_predicted,latitude_predicted
predictions_df_copy.to_csv('predictions_LSTM_november.csv', columns=['ID', 'longitude_predicted', 'latitude_predicted'], index=False)